### setup

In [1]:
!pip install -U sentence-transformers

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 85 kB 1.0 MB/s eta 0:00:011
     |████████████████████████████████| 7.9 MB 3.9 MB/s eta 0:00:01
     |████████████████████████████████| 6.8 MB 32.5 MB/s eta 0:00:01
     |████████████████████████████████| 311 kB 47.0 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 29.4 MB/s eta 0:00:01
     |████████████████████████████████| 3.8 MB 28.9 MB/s eta 0:00:01
     |████████████████████████████████| 670.2 MB 33.2 MB/s eta 0:00:01
     |████████████████████████████████| 196.0 MB 32.0 MB/s eta 0:00:01


     |████████████████████████████████| 99 kB 43.4 MB/s eta 0:00:01
     |████████████████████████████████| 56.5 MB 62.8 MB/s eta 0:00:01
     |████████████████████████████████| 823 kB 30.5 MB/s eta 0:00:01
     |████████████████████████████████| 731.7 MB 62.6 MB/s eta 0:00:01
     |████████████████████████████████| 121.6 MB 27.5 MB/s eta 0:00:01
     |████████████████████████████████| 23.7 MB 23.6 MB/s eta 0:00:01
     |████████████████████████████████| 209.8 MB 59.6 MB/s eta 0:00:01
     |████████████████████████████████| 14.1 MB 36.8 MB/s eta 0:00:01
     |████████████████████████████████| 124.2 MB 33.9 MB/s eta 0:00:01
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
     |████████████████████████████████| 6.9 MB 25.9 MB/s eta 0:00:01
     |████████████████████████████████| 33.8 MB 44.1 MB/s eta 0:00:01
     |████████████████████████████████| 33.8 MB 38.9 MB/s eta 0:00:01
     |██████

In [2]:
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 521 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 115 kB 32.6 MB/s eta 0:00:01
     |████████████████████████████████| 194 kB 38.6 MB/s eta 0:00:01
     |████████████████████████████████| 38.2 MB 43.6 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 41.4 MB/s eta 0:00:01
     |████████████████████████████████| 132 kB 45.0 MB/s eta 0:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.59.0
    Uninstalling tqdm-4.59.0:
      Successfully uninstalled tqdm-4.59.0


## intro

https://huggingface.co/sentence-transformers/all-mpnet-base-v2

https://huggingface.co/datasets/ms_marco?library=true

In [4]:
!export CUDA_VISIBLE_DEVICES=1 

In [5]:
import numpy as np

In [6]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device="cuda")
embeddings = model.encode(sentences)
print(embeddings)

[[ 0.0225026  -0.07829171 -0.02303072 ... -0.00827928  0.02652686
  -0.00201899]
 [ 0.04170236  0.00109741 -0.01553419 ... -0.02181628 -0.06359358
  -0.00875288]]


In [7]:
from datasets import load_dataset

dataset = load_dataset("ms_marco",  ['v1.1', 'v2.1'][0])

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

In [8]:
def make_ce_query(query, passage):
    return passage + "\n" + query

def get_data(t = "train", max_r = 10, encode=True):
    ps = list()
    qs = list()
    ms = list()
    qms = list()

    psd = dict()
    psc = 0
    data = list()
    for i, row in enumerate(dataset[t]):
        if i == max_r:
            break
        # print(row)

        qs.append(row["query"])
        ms.append(row["passages"]["is_selected"])
        texts = list()
        for passage in row["passages"]["passage_text"]:
            psc += 1
            if passage not in psd:
                idx = len(ps)
                psd[passage] = idx
                ps.append(passage)
            texts.append(
                make_ce_query(row["query"], passage)
            )

        data.append({
            "mask": row["passages"]["is_selected"],
            "texts": texts,
            "embs": model.encode(texts) if encode else texts
        })

        if encode:
            qms.extend(data[-1]["embs"])

    print(len(ps), len(dataset["train"]), psc)
    return data, qs, ps, ms, qms

In [7]:
%%time
data, qs, ps, ms, qms = get_data("train", 1000)

8223 82326 8251
CPU times: user 20min 34s, sys: 30.4 s, total: 21min 5s
Wall time: 1min 35s


In [8]:
len(qs), len(qms)

(1000, 8251)

In [9]:
def get_neg_qms(qms, qs, ps, ms):
    neg_qms_ = list()

    for i, q_ in enumerate(qs):
        for m_ in ms[i]:
            neg_qms_.append(
                make_ce_query(
                    q_,
                    ps[np.random.randint(0, len(ps))]
                )
            )
            
    neg_qms = model.encode(neg_qms_)
    
    return neg_qms

In [10]:
%%time
neg_qms = get_neg_qms(qms, qs, ps, ms)

CPU times: user 6min 22s, sys: 36 s, total: 6min 58s
Wall time: 48.7 s


In [11]:
len(qms), len(neg_qms)

(8251, 8251)

In [12]:
neg_qms_mean = neg_qms.mean(0)
qms_mean = np.array(qms).mean(0)
neg_qms_mean.shape, qms_mean.shape

((768,), (768,))

In [13]:
(((neg_qms_mean - neg_qms) ** 2).sum(-1) < ((qms_mean - neg_qms) ** 2).sum(-1)).mean()

0.6522845715670827

In [14]:
(((neg_qms_mean - np.array(qms)) ** 2).sum(-1) > ((qms_mean - np.array(qms)) ** 2).sum(-1)).mean()

0.6886438007514241

In [15]:
(((neg_qms_mean * neg_qms) ** 1).sum(-1) > ((qms_mean * neg_qms) ** 1).sum(-1)).mean()

0.6967640286025937

In [16]:
(((neg_qms_mean * np.array(qms)) ** 1).sum(-1) < ((qms_mean * np.array(qms)) ** 1).sum(-1)).mean()

0.645376318022058

In [17]:
(-0.5)**1, 4**1, neg_qms_mean.shape

(-0.5, 4, (768,))

In [18]:
%%time
data_v, qs_v, ps_v, ms_v, qms_v = get_data("validation", 1000)

8218 82326 8243
CPU times: user 21min 41s, sys: 38.2 s, total: 22min 20s
Wall time: 1min 46s


In [19]:
neg_qms_v = get_neg_qms(qms_v, qs_v, ps_v, ms_v)

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(768, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.layers(x)

In [34]:
mlp = MLP()
mlp.to("cuda")
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

In [35]:
%%time
mean_train_losses = []
mean_valid_losses = []
valid_acc_list = []

epochs = 40
reneg_each = 5

bs = 32
best = 0

qms_tensor = torch.Tensor(np.array(qms)).to("cuda")
neg_qms_tensor = torch.Tensor(neg_qms).to("cuda")

qms_v_tensor = torch.Tensor(np.array(qms_v)).to("cuda")
neg_qms_v_tensor = torch.Tensor(neg_qms_v).to("cuda")

CE_MLP_PATH = "ce_mlp.torch"

mlp.train()
for epoch in range(epochs):
    shuffle = np.arange(len(qms))
    np.random.shuffle(shuffle)

    for i in range(0, len(qms), bs):
        optimizer.zero_grad()

        idx = np.arange(i, min(i + bs, len(qms)))
        idx = shuffle[idx]

        good_emb = qms_tensor[idx]
        bad_emb = neg_qms_tensor[idx]

        loss = (
            loss_fn(mlp(good_emb), torch.Tensor([[1]] * len(idx)).to("cuda"))
            + 
            loss_fn(mlp(bad_emb), torch.Tensor([[0]] * len(idx)).to("cuda"))
        )

        loss.backward()
        optimizer.step()

        # print(loss)
        
    th = 0.5
    print("epoch = ", epoch)
    print("Train Positive Accuracy = ", torch.mean((mlp(qms_tensor) > th), dtype=float))
    print("Train Negative Accuracy = ", torch.mean((mlp(neg_qms_tensor) < th), dtype=float))
    
    pa = torch.mean((mlp(qms_v_tensor) > th), dtype=float)
    na = torch.mean((mlp(neg_qms_v_tensor) < th), dtype=float)
    
    print("Val Positive Accuracy = ", pa)
    print("Val Negative Accuracy = ", na)

    if pa + na > best:
        torch.save(mlp, CE_MLP_PATH)
        print("Saved....")
        best = pa + na
        
    if epoch + 1 != epochs:
        if (epoch + 1) % reneg_each == 0:
            print("Regen Negatives...")
            neg_qms = get_neg_qms(qms, qs, ps, ms)
            neg_qms_tensor = torch.Tensor(neg_qms).to("cuda")
            
mlp.eval()

epoch =  0
Train Positive Accuracy =  tensor(0.8067, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.6804, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.7984, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.6328, device='cuda:0', dtype=torch.float64)
Saved....
epoch =  1
Train Positive Accuracy =  tensor(0.7770, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.7629, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.7622, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.7054, device='cuda:0', dtype=torch.float64)
Saved....
epoch =  2
Train Positive Accuracy =  tensor(0.8201, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.7445, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.7813, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.6785, device='cuda:0', dtype=torch.float64)

epoch =  25
Train Positive Accuracy =  tensor(0.9935, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.8793, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.7572, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.7634, device='cuda:0', dtype=torch.float64)
epoch =  26
Train Positive Accuracy =  tensor(0.9750, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.9535, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.6987, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.8131, device='cuda:0', dtype=torch.float64)
epoch =  27
Train Positive Accuracy =  tensor(0.9928, device='cuda:0', dtype=torch.float64)
Train Negative Accuracy =  tensor(0.9320, device='cuda:0', dtype=torch.float64)
Val Positive Accuracy =  tensor(0.7369, device='cuda:0', dtype=torch.float64)
Val Negative Accuracy =  tensor(0.7748, device='cuda:0', dtype=torch.float64)
epoch =  28
Trai

MLP(
  (layers): Sequential(
    (0): Linear(in_features=768, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [215]:
torch.save(mlp, "mlp.torch")

In [173]:
th = 0.5
(mlp(torch.Tensor(np.array(qms))) > th).numpy().mean()

0.8115379953944977

In [174]:
(mlp(torch.Tensor(neg_qms)) < th).numpy().mean()

0.6652526966428312

In [159]:
mlp(torch.Tensor(neg_qms)).mean(), mlp(torch.Tensor(np.array(qms))).mean()

(tensor(0.4050, grad_fn=<MeanBackward0>),
 tensor(0.6830, grad_fn=<MeanBackward0>))

In [35]:
%%time
data_t, qs_t, ps_t, ms_t, qms_t = get_data("test", 1000)

8201 82326 8238
CPU times: user 16min 52s, sys: 28.8 s, total: 17min 21s
Wall time: 1min 12s


In [38]:
%%time
neg_qms_t = get_neg_qms(qms_t, qs_t, ps_t)

CPU times: user 4min 10s, sys: 29.3 s, total: 4min 39s
Wall time: 37.9 s


In [115]:
(mlp(torch.Tensor(np.array(qms_t))) > 0.5).numpy().mean()

0.7671764991502792

In [116]:
(mlp(torch.Tensor(neg_qms_t)) < 0.5).numpy().mean()

0.7146151978635591

In [36]:
import tqdm

def get_all_texts(qms, qs, ps, I=int(1e9)):
    ces = list()
    for q_i in tqdm.tqdm_notebook(range(min(len(qs), I))):
        row = [
            make_ce_query(qs[q_i], ps[p_i])
            for p_i in range(len(ps))
        ]
        
        
        row = model.encode(row, convert_to_tensor=True)
        ces.append(mlp(row).cpu().detach().numpy())
        
    print(len(qs), len(ps))
    return ces

In [94]:
mlp.to("cuda")
all_ = get_all_texts(qms, qs, ps, I = 10)
len(all_)

<ipython-input-36-c3e025bbd26a>:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for q_i in tqdm.tqdm_notebook(range(min(len(qs), I))):


  0%|          | 0/10 [00:00<?, ?it/s]

1000 8223


10

In [97]:
psd = dict()
max_r = 10
q2ans = list()
for i, row in enumerate(dataset["train"]):
    if i == max_r:
        break
    q2ans.append(list())
    for passage in row["passages"]["passage_text"]:
        if passage not in psd:
            idx = len(psd)
            psd[passage] = idx
        q2ans[-1].append(psd[passage])

iss = list()
for qi_ in range(max_r):
    iss.append(len(set(q2ans[qi_]).intersection(all_[qi_].flatten().argsort()[::-1][:100])) / len(set(q2ans[qi_])))
np.mean(iss), iss

(0.5684920634920636,
 [0.0,
  0.8571428571428571,
  0.5,
  0.3333333333333333,
  0.25,
  0.3,
  1.0,
  0.8888888888888888,
  0.8888888888888888,
  0.6666666666666666])

In [98]:
qs_emb = model.encode(qs)

In [101]:
ps_emb = model.encode(ps)

array([   9,    7,    5,    6,    8,    1,    4,    0,    3, 6041, 3668,
       5842, 3097, 5250, 6899,  851, 4909, 6043, 4602, 5836, 5840, 3675,
       6902,    2, 7074,  221, 4908, 3669, 5382, 7387, 1003,  594, 5835,
       1133, 1001, 2069, 4907, 3815,  222, 6690, 3005, 2527, 5257, 5376,
        217, 2497,  853,  433, 3369,  759, 8152, 6805, 4890,  170,  248,
        852,  365, 4658, 6801, 4891, 3676,  173,  245, 4721, 1132, 5843,
       4489, 4659, 7500, 4341, 1351, 2851, 7393, 7389, 6903, 4895,  850,
       2641, 7092, 6608, 1889, 3095, 1625, 7497, 5292, 7853, 7499, 4894,
       3134, 3577, 3096, 5814,  857, 3832, 4488, 2500, 2523, 3582, 4493,
        849])

In [108]:
psd = dict()
max_r = 1000
q2ans = list()
for i, row in enumerate(dataset["train"]):
    if i == max_r:
        break
    q2ans.append(list())
    for passage in row["passages"]["passage_text"]:
        if passage not in psd:
            idx = len(psd)
            psd[passage] = idx
        q2ans[-1].append(psd[passage])

iss = list()
for qi_ in range(max_r):
    pred = np.argsort(ps_emb @ qs_emb[qi_])[::-1][:len(q2ans[qi_])]
    iss.append(len(set(q2ans[qi_]).intersection(pred)) / len(set(q2ans[qi_])))
np.mean(iss), iss

(0.8972194444444446,
 [0.9,
  1.0,
  0.9,
  1.0,
  0.5,
  0.9,
  1.0,
  0.8888888888888888,
  0.8888888888888888,
  1.0,
  0.8888888888888888,
  1.0,
  0.25,
  1.0,
  0.8,
  0.6666666666666666,
  0.9,
  0.8571428571428571,
  0.9,
  0.8333333333333334,
  1.0,
  1.0,
  0.625,
  0.75,
  0.7777777777777778,
  1.0,
  1.0,
  1.0,
  1.0,
  0.75,
  1.0,
  0.7142857142857143,
  1.0,
  1.0,
  1.0,
  0.6,
  1.0,
  1.0,
  0.875,
  1.0,
  0.9,
  1.0,
  1.0,
  0.8,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  0.8,
  0.7777777777777778,
  0.6,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  0.875,
  1.0,
  0.7142857142857143,
  1.0,
  1.0,
  0.2,
  0.7777777777777778,
  1.0,
  1.0,
  1.0,
  0.8333333333333334,
  0.7777777777777778,
  1.0,
  0.6666666666666666,
  0.875,
  0.875,
  0.875,
  1.0,
  0.875,
  1.0,
  1.0,
  0.8888888888888888,
  0.875,
  0.75,
  0.9,
  0.5555555555555556,
  1.0,
  0.8571428571428571,
  1.0,
  1.0,
  0.875,
  0.8,
  1.0,
  1.0,
  0.8888888888888888,
  0.8,
  1.0,
  1.0,
  1.0,
 

In [112]:
R1000 = qs_emb @ ps_emb.T

In [113]:
R1000

array([[ 0.44760713,  0.51731   ,  0.27097726, ..., -0.01348421,
        -0.04682447, -0.03000023],
       [ 0.00190526,  0.04977059, -0.03790381, ..., -0.08634734,
        -0.09433351, -0.1000261 ],
       [ 0.11997095,  0.17196429,  0.12033795, ...,  0.06874324,
         0.12098464,  0.09233715],
       ...,
       [-0.05174239, -0.04145098, -0.03925405, ...,  0.01748889,
        -0.01697676,  0.02611255],
       [-0.06159192, -0.04293635,  0.00459509, ...,  0.20471342,
         0.18004717,  0.13578539],
       [-0.05425338, -0.01935816, -0.0282955 , ...,  0.8354984 ,
         0.82424223,  0.7680844 ]], dtype=float32)

In [116]:
np.savez_compressed('./R1000train.npz', R=R1000, Q2ANS=q2ans)

/home/shevkunov/anaconda3/lib/python3.8/site-packages/numpy/core/_asarray.py:136: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  return array(a, dtype, copy=False, order=order, subok=True)


In [90]:
X = 0
print(qs[X])

for i in all_[X].flatten().argsort()[::-1][:100]:
    print(ps[i], "\n\n\n")

what is rba
Howard Carter. Howard Carter (1874-1939), the renowned archaeologist who discovered the tomb of King Tutankhamun, was a private and stubborn man whose perfectionism frequently got him into trouble. COPYRIGHT 2004 The Gale Group Inc. Howard Carter (1874-1939), the renowned archaeologist who discovered the tomb of King Tutankhamun, was a private and stubborn man whose perfectionism frequently got him into trouble. 



Generalized epilepsy is primary because the epilepsy is the originally diagnosed condition itself, as opposed to secondary epilepsy, which occurs as a symptom of a diagnosed condition. Generalized seizures can be either absence seizures, myoclonic seizures, clonic seizures, tonic-clonic seizures or atonic seizures. 



Typically, if you want help applying for probate, there are three main providers, banks, wills and probate companies, and Solicitors. Most banks in our experience charge a fixed percentage of the estate typically, 2-4%, which we believe to be unfa

In [117]:
len(q2ans)

1000

####

In [118]:
%%time
data, qs, ps, ms, qms = get_data("train", 2000)

16433 82326 16511
CPU times: user 36min 12s, sys: 36 s, total: 36min 48s
Wall time: 2min 29s


In [119]:
qs_emb = model.encode(qs)

In [120]:
ps_emb = model.encode(ps)

In [121]:
psd = dict()
max_r = 2000
q2ans = list()
for i, row in enumerate(dataset["train"]):
    if i == max_r:
        break
    q2ans.append(list())
    for passage in row["passages"]["passage_text"]:
        if passage not in psd:
            idx = len(psd)
            psd[passage] = idx
        q2ans[-1].append(psd[passage])

iss = list()
for qi_ in range(max_r):
    pred = np.argsort(ps_emb @ qs_emb[qi_])[::-1][:len(q2ans[qi_])]
    iss.append(len(set(q2ans[qi_]).intersection(pred)) / len(set(q2ans[qi_])))
np.mean(iss), iss

(0.8537172619047619,
 [0.9,
  1.0,
  0.9,
  0.7777777777777778,
  0.5,
  0.8,
  1.0,
  0.8888888888888888,
  0.8888888888888888,
  1.0,
  0.8888888888888888,
  0.9,
  0.25,
  1.0,
  0.8,
  0.5555555555555556,
  0.9,
  0.8571428571428571,
  0.9,
  0.8333333333333334,
  1.0,
  0.875,
  0.625,
  0.375,
  0.6666666666666666,
  1.0,
  0.7142857142857143,
  1.0,
  1.0,
  0.75,
  0.8888888888888888,
  0.7142857142857143,
  1.0,
  1.0,
  0.8571428571428571,
  0.6,
  1.0,
  1.0,
  0.875,
  1.0,
  0.8,
  1.0,
  1.0,
  0.7,
  0.8888888888888888,
  1.0,
  0.7777777777777778,
  1.0,
  1.0,
  1.0,
  1.0,
  0.7,
  0.7777777777777778,
  0.6,
  1.0,
  1.0,
  1.0,
  0.6666666666666666,
  1.0,
  0.875,
  0.6666666666666666,
  0.7142857142857143,
  0.8333333333333334,
  1.0,
  0.0,
  0.5555555555555556,
  1.0,
  0.8888888888888888,
  1.0,
  0.8333333333333334,
  0.7777777777777778,
  1.0,
  0.6666666666666666,
  0.875,
  0.875,
  0.875,
  1.0,
  0.875,
  1.0,
  1.0,
  0.8888888888888888,
  0.625,
  0.75,


In [122]:
R2000 = qs_emb @ ps_emb.T
np.savez_compressed('./R2000train.npz', R=R2000, Q2ANS=q2ans)

In [123]:
dataset["test"]

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 9650
})

In [124]:
R2000

array([[ 0.4476072 ,  0.51730996,  0.27097738, ...,  0.07404932,
         0.05676902,  0.07167948],
       [ 0.00190527,  0.04977061, -0.03790371, ...,  0.07161373,
         0.10618924,  0.11791737],
       [ 0.11997091,  0.17196432,  0.12033798, ..., -0.08548873,
        -0.0755754 , -0.07016385],
       ...,
       [ 0.01123581, -0.01151523,  0.06818046, ..., -0.11696956,
        -0.14415547, -0.16731963],
       [-0.05960517, -0.0086166 , -0.02945979, ..., -0.09281544,
        -0.11510713, -0.10211937],
       [ 0.04833945,  0.03932492, -0.04429494, ...,  0.63032943,
         0.6076563 ,  0.7501704 ]], dtype=float32)

In [9]:
def getR_K(t, k):
    data, qs, ps, ms, qms = get_data(t, k)
    qs_emb = model.encode(qs)
    ps_emb = model.encode(ps)
    
    psd = dict()
    max_r = k
    q2ans = list()
    for i, row in enumerate(dataset[t]):
        if i == max_r:
            break
        q2ans.append(list())
        for passage in row["passages"]["passage_text"]:
            if passage not in psd:
                idx = len(psd)
                psd[passage] = idx
            q2ans[-1].append(psd[passage])

    iss = list()
    for qi_ in range(max_r):
        pred = np.argsort(ps_emb @ qs_emb[qi_])[::-1][:len(q2ans[qi_])]
        iss.append(len(set(q2ans[qi_]).intersection(pred)) / len(set(q2ans[qi_])))
    print("np.mean(iss) = ", np.mean(iss))
    
    R = qs_emb @ ps_emb.T
    np.savez_compressed(f'./R{k}{t}.npz', R=R, Q2ANS=q2ans)

In [126]:
getR_K("train", 3000)

24625 82326 24772
np.mean(iss) =  0.8264592592592592


In [127]:
getR_K("test", 3000)

24493 82326 24650
np.mean(iss) =  0.8201563492063492


In [128]:
getR_K("test", 5000)

40718 82326 41100
np.mean(iss) =  0.7759951587301587


In [10]:
dataset["test"]

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 9650
})

In [11]:
getR_K("test", 7000)

56790 82326 57474
np.mean(iss) =  0.7481175736961451


/home/shevkunov/anaconda3/lib/python3.8/site-packages/numpy/core/_asarray.py:136: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  return array(a, dtype, copy=False, order=order, subok=True)


In [12]:
getR_K("test", 10000)

77897 82326 79176


IndexError: index 9650 is out of bounds for axis 0 with size 9650

In [ ]:
getR_K("test", 9650)

77897 82326 79176
